# Road Accident Severity Prediction using Machine Learning

## Phase 6 — Feature Selection & Data Preparation

---

**Project:** Road Accident Severity Prediction using Machine Learning
**Institution:** On-Campus Research Internship, IIIT Vadodara
**Notebook:** `05_Feature_Selection_and_Data_Preparation.ipynb`
**Phase:** 6 of N — Feature Selection & Data Preparation
**Input Dataset:** `Dataset/processed/featured_accident_data.csv`
**Target Variable:** `Accident_Severity`

---

### Objective of this Notebook

This notebook takes the engineered dataset from Phase 5 and prepares it for machine
learning: separating features from the target, removing columns unsuitable for
prediction, encoding categorical variables, analyzing and appropriately handling class
imbalance, splitting into train/test sets, and saving the final model-ready arrays.

Specifically, this notebook:

1. Loads and re-verifies `featured_accident_data.csv`.
2. Separates the feature matrix `X` from the target `y` and verifies target
   distribution.
3. Removes identifier-like and redundant columns unsuitable for prediction, with an
   explicit justification for every removal.
4. Encodes categorical variables using a cardinality-aware strategy suitable for both
   tree-based and linear model families.
5. Analyzes class imbalance and discusses appropriate handling strategies.
6. Performs a stratified 80/20 train-test split (`random_state=42`).
7. Applies resampling to the **training set only** (never the test set), to correctly
   address class imbalance without leaking information.
8. Saves the final `X_train`, `X_test`, `y_train`, `y_test` arrays to
   `Dataset/processed/ml_ready/`.
9. Summarizes every retained feature, its type, encoding, and rationale.
10. Performs a final validation of shapes, feature count, and target distribution.

> **Scope restriction:** This notebook performs **feature selection and data
> preparation only**. It does **not** train, evaluate, or tune any machine learning
> model — that begins in the next phase.


---
## Setup — Import Libraries & Configure Environment

**Purpose:** Import the libraries required for feature selection and data preparation,
and configure pandas display options for consistent, readable output.


In [1]:
# ---- Core Libraries ----
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

# ---- Environment Configuration ----
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 150)
pd.set_option("display.width", 150)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

TARGET_COLUMN = "Accident_Severity"
RANDOM_STATE = 42

# SMOTENC (categorical + continuous aware SMOTE) is used for imbalance handling,
# since standard SMOTE is not designed for one-hot / label-encoded categorical
# features. It is imported defensively, so this notebook still runs end-to-end
# (with a clear warning) even if 'imbalanced-learn' is not yet installed.
try:
    from imblearn.over_sampling import SMOTENC
    IMBLEARN_AVAILABLE = True
except ImportError:
    IMBLEARN_AVAILABLE = False
    print(
        "[WARNING] 'imbalanced-learn' is not installed. Install it with:\n"
        "    pip install imbalanced-learn --break-system-packages\n"
        "Class imbalance will be analyzed but resampling will be skipped until "
        "this package is available."
    )


def column_exists(df: pd.DataFrame, column: str) -> bool:
    '''
    Check whether a column exists in the dataframe, printing a clear
    skip-message if it does not, so every step degrades gracefully instead
    of raising a KeyError.

    Args:
        df (pd.DataFrame): The dataframe to check.
        column (str): Column name to check for.

    Returns:
        bool: True if the column exists, False otherwise.
    '''
    exists = column in df.columns
    if not exists:
        print(f"[SKIPPED] Column '{column}' was not found in the dataset.")
    return exists


print("Libraries imported and environment configured successfully.")
print(f"imbalanced-learn available: {IMBLEARN_AVAILABLE}")


[WARNING] 'imbalanced-learn' is not installed. Install it with:
    pip install imbalanced-learn --break-system-packages
Class imbalance will be analyzed but resampling will be skipped until this package is available.
Libraries imported and environment configured successfully.
imbalanced-learn available: False


**Interpretation**

- `pandas`, `numpy`, and `scikit-learn` cover the core data preparation and splitting
  needs; `imbalanced-learn` is used only for the class-imbalance step and is imported
  defensively so the notebook never crashes outright if it is missing from the
  environment.
- `column_exists()` is reused from earlier notebooks so every section can check its
  required columns before using them.


---
## 1. Load Dataset

**Purpose:** Load `Dataset/processed/featured_accident_data.csv` (produced in Phase 5)
using robust exception handling, consistent with prior notebooks.


In [2]:
FEATURED_DATA_PATH = Path("..") / "Dataset" / "processed" / "featured_accident_data.csv"

try:
    if not FEATURED_DATA_PATH.exists():
        raise FileNotFoundError(
            f"Featured dataset not found at: {FEATURED_DATA_PATH.resolve()}. "
            f"Please run 04_Feature_Engineering.ipynb first."
        )
    df = pd.read_csv(FEATURED_DATA_PATH, low_memory=False)
    print(f"Featured dataset loaded successfully from: {FEATURED_DATA_PATH.resolve()}")
    print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
except (FileNotFoundError, pd.errors.EmptyDataError, pd.errors.ParserError) as error:
    print(f"[ERROR] {error}")
    raise


Featured dataset loaded successfully from: C:\Users\Stalin\OneDrive\Desktop\draftrajproject\road-accident-severity-prediction\road-accident-severity-prediction\Dataset\processed\featured_accident_data.csv
Shape: 2,715,940 rows x 58 columns


**Interpretation**

- The dataset is loaded directly from the Phase 5 output with no modification, keeping
  this notebook a clean continuation of the pipeline.


---
## 2. Verify Dataset

**Purpose:** Re-confirm the dataset's shape, columns, data types, and missing values
before beginning feature selection.


In [3]:
print("Shape:", df.shape)
print()
print("Columns:")
for col in df.columns:
    print(f"  - {col}")


Shape: (2715940, 58)

Columns:
  - 1st_Road_Class
  - Accident_Severity
  - Date
  - Day_of_Week
  - Did_Police_Officer_Attend_Scene_of_Accident
  - Junction_Control
  - Junction_Detail
  - Latitude
  - Light_Conditions
  - Local_Authority_(District)
  - Longitude
  - Number_of_Vehicles
  - Pedestrian_Crossing-Human_Control
  - Pedestrian_Crossing-Physical_Facilities
  - Road_Surface_Conditions
  - Road_Type
  - Speed_limit
  - Time
  - Urban_or_Rural_Area
  - Weather_Conditions
  - Year_accident
  - InScotland
  - Age_Band_of_Driver
  - Age_of_Vehicle
  - Driver_Home_Area_Type
  - Engine_Capacity_.CC.
  - Journey_Purpose_of_Driver
  - Junction_Location
  - Propulsion_Code
  - Sex_of_Driver
  - Towing_and_Articulation
  - Vehicle_Leaving_Carriageway
  - Vehicle_Location.Restricted_Lane
  - Vehicle_Manoeuvre
  - Vehicle_Type
  - Was_Vehicle_Left_Hand_Drive
  - X1st_Point_of_Impact
  - Year_vehicle
  - Year
  - Month
  - Day
  - Quarter
  - Is_Weekend
  - Hour
  - Minute
  - Time_of_Day


In [4]:
dtype_table = pd.DataFrame(
    {"Column Name": df.columns, "Data Type": df.dtypes.astype(str).values}
)
dtype_table


,Column Name,Data Type
0,1st_Road_Class,object
1,Accident_Severity,object
2,Date,object
3,Day_of_Week,object
4,Did_Police_Officer_Attend_Scene_of_Accident,float64
5,Junction_Control,object
6,Junction_Detail,object
7,Latitude,float64
8,Light_Conditions,object
9,Local_Authority_(District),object


In [5]:
missing_summary = df.isnull().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)

if missing_summary.empty:
    print("No missing values present in the dataset.")
else:
    print(f"{len(missing_summary)} column(s) contain missing values:")
    display(missing_summary.to_frame("Missing Count"))


1 column(s) contain missing values:


,Missing Count
Date,1632422


**Interpretation**

- This verification mirrors the checks performed in every previous notebook and
  confirms the dataset is in the expected state — carrying forward all original
  cleaned columns plus the 19 engineered features from Phase 5 — before any columns
  are removed or encoded.


---
## 3. Target Variable

**Purpose:** Separate the feature matrix `X` from the target vector `y`, and verify
the target's class distribution before any further preparation.


In [6]:
y = df[TARGET_COLUMN].copy()
X = df.drop(columns=[TARGET_COLUMN]).copy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")


X shape: (2715940, 57)
y shape: (2715940,)


In [7]:
target_counts = y.value_counts()
target_percentages = y.value_counts(normalize=True) * 100

target_summary = pd.DataFrame(
    {
        "Severity": target_counts.index,
        "Count": target_counts.values,
        "Percentage (%)": target_percentages.values.round(2),
    }
)
target_summary


,Severity,Count,Percentage (%)
0,Slight,2315841,85.27
1,Serious,364619,13.43
2,Fatal,35480,1.31


**Interpretation**

- `X` now holds every predictor column, and `y` holds only `Accident_Severity`.
- The target distribution reconfirms the class imbalance already identified during
  Phase 4's EDA (typically dominated by `Slight`, with `Serious` and `Fatal` as
  progressively smaller minority classes) — this is examined in depth, and acted on,
  in Section 6 below.


---
## 4. Remove Columns

**Purpose:** Remove columns from `X` that should never be used for prediction —
identifier-like or non-generalizable fields, and raw columns fully superseded by an
engineered equivalent — with an explicit justification for every removal.

**Categories of columns removed:**

| Category | Columns | Justification |
|----------|---------|----------------|
| **Identifier / Non-generalizable** | `Date`, `Time` | Already fully decomposed into `Year`, `Month`, `Day`, `Quarter`, `Is_Weekend`, `Hour`, `Minute`, `Time_of_Day` in Phase 5; the raw values themselves are too granular (near-unique per record) to generalize to unseen dates/times. |
| **Identifier / Non-generalizable** | `Local_Authority_(District)` | A very high-cardinality administrative code; using it directly risks the model memorizing per-district quirks rather than learning generalizable severity patterns. |
| **Identifier / Non-generalizable** | `Latitude`, `Longitude` | Precise geographic coordinates function as near-unique location identifiers at this granularity; retaining them risks overfitting to specific accident spots rather than generalizable spatial patterns (a deliberately simple, non-geospatial baseline feature set is used here). |
| **Redundant (superseded by an engineered feature)** | `Weather_Conditions`, `Light_Conditions`, `Road_Type`, `Vehicle_Type`, `Urban_or_Rural_Area`, `Junction_Detail` | Each of these raw, high-cardinality text columns was explicitly grouped into a cleaner engineered equivalent in Phase 5 (`Weather_Group`, `Light_Group`, `Road_Category`, `Vehicle_Type_Group`, `Urban_Rural_Group`, `Junction_Group` respectively); keeping both would duplicate the same underlying signal. |

Additionally, a **redundancy audit** is run programmatically below to catch any
columns that are perfect (or near-perfect) duplicates of another column — an
important safety net beyond the manually curated list above.


In [8]:
IDENTIFIER_NON_GENERALIZABLE_COLUMNS = [
    "Date",
    "Time",
    "Local_Authority_(District)",
    "Latitude",
    "Longitude",
]

REDUNDANT_SUPERSEDED_COLUMNS = [
    "Weather_Conditions",
    "Light_Conditions",
    "Road_Type",
    "Vehicle_Type",
    "Urban_or_Rural_Area",
    "Junction_Detail",
]

COLUMN_REMOVAL_REASONS = {
    **{col: "Identifier / non-generalizable — already decomposed or too granular to generalize." for col in IDENTIFIER_NON_GENERALIZABLE_COLUMNS},
    **{col: "Redundant — fully superseded by an engineered grouped feature from Phase 5." for col in REDUNDANT_SUPERSEDED_COLUMNS},
}

candidate_removals = IDENTIFIER_NON_GENERALIZABLE_COLUMNS + REDUNDANT_SUPERSEDED_COLUMNS
columns_actually_removed = [col for col in candidate_removals if col in X.columns]
columns_not_found = [col for col in candidate_removals if col not in X.columns]

shape_before_removal = X.shape
X = X.drop(columns=columns_actually_removed)
shape_after_removal = X.shape

print(f"Columns before removal : {shape_before_removal[1]}")
print(f"Columns removed        : {len(columns_actually_removed)}")
print(f"Columns after removal  : {shape_after_removal[1]}")
print()
print("Removed Columns and Justification:")
for col in columns_actually_removed:
    print(f"  - {col}: {COLUMN_REMOVAL_REASONS[col]}")

if columns_not_found:
    print()
    print("Note: The following candidate columns were not present and were skipped safely:")
    for col in columns_not_found:
        print(f"  - {col}")


Columns before removal : 57
Columns removed        : 11
Columns after removal  : 46

Removed Columns and Justification:
  - Date: Identifier / non-generalizable — already decomposed or too granular to generalize.
  - Time: Identifier / non-generalizable — already decomposed or too granular to generalize.
  - Local_Authority_(District): Identifier / non-generalizable — already decomposed or too granular to generalize.
  - Latitude: Identifier / non-generalizable — already decomposed or too granular to generalize.
  - Longitude: Identifier / non-generalizable — already decomposed or too granular to generalize.
  - Weather_Conditions: Redundant — fully superseded by an engineered grouped feature from Phase 5.
  - Light_Conditions: Redundant — fully superseded by an engineered grouped feature from Phase 5.
  - Road_Type: Redundant — fully superseded by an engineered grouped feature from Phase 5.
  - Vehicle_Type: Redundant — fully superseded by an engineered grouped feature from Phase 5.
 

In [9]:
def find_duplicate_columns(df: pd.DataFrame) -> list:
    '''
    Identify pairs of columns that are perfect duplicates of one another
    (identical values in every row), as a redundancy safety net beyond
    manually curated removal lists.

    Args:
        df (pd.DataFrame): The dataframe to audit.

    Returns:
        list: A list of (column_a, column_b) tuples for duplicate pairs found.
    '''
    duplicate_pairs = []
    columns = df.columns.tolist()

    for i in range(len(columns)):
        for j in range(i + 1, len(columns)):
            col_a, col_b = columns[i], columns[j]
            try:
                if df[col_a].equals(df[col_b]):
                    duplicate_pairs.append((col_a, col_b))
            except (TypeError, ValueError):
                continue

    return duplicate_pairs


duplicate_column_pairs = find_duplicate_columns(X)

print(f"Duplicate column pairs found: {len(duplicate_column_pairs)}")
for col_a, col_b in duplicate_column_pairs:
    print(f"  - '{col_a}' is identical to '{col_b}'")


Duplicate column pairs found: 1
  - 'Age_of_Vehicle' is identical to 'Vehicle_Age'


In [10]:
# For every duplicate pair found, drop the second (redundant) column, keeping
# the first — which, by construction in Phase 5, is typically the more clearly
# named engineered version (e.g., keep 'Vehicle_Age', drop 'Age_of_Vehicle').
duplicate_columns_to_drop = [col_b for _, col_b in duplicate_column_pairs]

if duplicate_columns_to_drop:
    print("Dropping redundant duplicate columns identified by the audit:")
    for col in duplicate_columns_to_drop:
        print(f"  - {col}")
    X = X.drop(columns=duplicate_columns_to_drop)
else:
    print("No exact duplicate columns found — no additional columns dropped.")

print(f"\nFeature count after redundancy audit: {X.shape[1]}")


Dropping redundant duplicate columns identified by the audit:
  - Vehicle_Age

Feature count after redundancy audit: 45


**Interpretation**

- Every column removed above falls into one of two explicit categories — identifier /
  non-generalizable, or redundant with an engineered equivalent — and none of them
  constitutes direct target leakage (that risk, e.g. `Number_of_Casualties`, was
  already addressed in Phase 3).
- The programmatic duplicate-column audit is a useful safety net: for example,
  `Vehicle_Age` and `Age_of_Vehicle` (created in Phase 5 as a renamed copy) are exact
  duplicates of one another, and this audit automatically detects and removes the
  redundant copy without needing to hardcode that specific pair in advance.
- The removal logic checks for column presence before dropping, so it will not raise a
  `KeyError` if a particular column is absent in your specific run (e.g., if it was
  already dropped in Phase 3 due to excessive missingness).


---
## 5. Encode Categorical Variables

**Purpose:** Automatically detect categorical columns in `X` and encode them using a
strategy suitable for **both** tree-based and linear model families, without using any
target-dependent encoding (which would risk leaking target information before the
train-test split).

**Encoding Strategy (cardinality-based, non-target-dependent):**

| Cardinality | Encoding Applied | Why |
|-------------|-------------------|-----|
| Low (≤ `ONE_HOT_CARDINALITY_THRESHOLD` unique values) | **One-Hot Encoding** | Produces a small number of binary columns with no artificial ordinal relationship — ideal for linear models, and perfectly usable (if slightly less compact) for tree-based models. |
| High (> `ONE_HOT_CARDINALITY_THRESHOLD` unique values) | **Ordinal / Label Encoding** | Avoids an explosion of one-hot columns for high-cardinality fields; tree-based models can split on arbitrary integer codes effectively, though linear models should be used with more caution on these particular columns. |

Both encoding methods used here depend only on the **feature values themselves**, not
on the target `y`, so it is safe to apply them to the full dataset before the
train-test split (unlike target encoding, which must only ever be fit on training
data).


In [11]:
ONE_HOT_CARDINALITY_THRESHOLD = 15

categorical_columns = X.select_dtypes(include=["object", "category"]).columns.tolist()

# Snapshot of X immediately before one-hot expansion, used later in Section 9 to
# build a feature summary at the readable, pre-expansion conceptual-feature level.
pre_encoding_columns_snapshot = X.copy()

print(f"Categorical columns detected: {len(categorical_columns)}")
for col in categorical_columns:
    print(f"  - {col}: {X[col].nunique()} unique values")


Categorical columns detected: 26
  - 1st_Road_Class: 6 unique values
  - Day_of_Week: 7 unique values
  - Junction_Control: 6 unique values
  - Road_Surface_Conditions: 6 unique values
  - InScotland: 2 unique values
  - Age_Band_of_Driver: 13 unique values
  - Driver_Home_Area_Type: 5 unique values
  - Journey_Purpose_of_Driver: 9 unique values
  - Junction_Location: 11 unique values
  - Propulsion_Code: 13 unique values
  - Sex_of_Driver: 5 unique values
  - Towing_and_Articulation: 8 unique values
  - Vehicle_Leaving_Carriageway: 11 unique values
  - Vehicle_Manoeuvre: 20 unique values
  - Was_Vehicle_Left_Hand_Drive: 4 unique values
  - X1st_Point_of_Impact: 7 unique values
  - Month: 12 unique values
  - Time_of_Day: 4 unique values
  - Speed_Category: 3 unique values
  - Road_Category: 4 unique values
  - Junction_Group: 2 unique values
  - Weather_Group: 6 unique values
  - Light_Group: 4 unique values
  - Urban_Rural_Group: 3 unique values
  - Vehicle_Type_Group: 5 unique value

In [12]:
def encode_categorical_features(
    df: pd.DataFrame, categorical_columns: list, cardinality_threshold: int
) -> tuple:
    '''
    Encode categorical columns using a cardinality-based strategy:
    one-hot encoding for low-cardinality columns, ordinal/label encoding
    for high-cardinality columns. Both approaches depend only on the
    feature values, not the target, so they are safe to apply before a
    train-test split.

    Args:
        df (pd.DataFrame): The dataframe containing the categorical columns.
        categorical_columns (list): Names of categorical columns to encode.
        cardinality_threshold (int): Maximum unique values for one-hot encoding;
                                      columns above this are label-encoded instead.

    Returns:
        tuple:
            - pd.DataFrame: The dataframe with categorical columns encoded.
            - list: Names of columns that were one-hot encoded (before expansion).
            - list: Names of columns that were label/ordinal encoded.
            - dict: Mapping of label-encoded column name -> fitted OrdinalEncoder.
    '''
    df = df.copy()
    one_hot_columns = []
    label_encoded_columns = []
    ordinal_encoders = {}

    low_cardinality_columns = [
        col for col in categorical_columns if df[col].nunique() <= cardinality_threshold
    ]
    high_cardinality_columns = [
        col for col in categorical_columns if df[col].nunique() > cardinality_threshold
    ]

    if low_cardinality_columns:
        df = pd.get_dummies(df, columns=low_cardinality_columns, drop_first=True)
        one_hot_columns = low_cardinality_columns

    for col in high_cardinality_columns:
        encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        df[col] = encoder.fit_transform(df[[col]].astype(str))
        ordinal_encoders[col] = encoder
        label_encoded_columns.append(col)

    return df, one_hot_columns, label_encoded_columns, ordinal_encoders


X, one_hot_encoded_columns, label_encoded_columns, ordinal_encoders = encode_categorical_features(
    X, categorical_columns, ONE_HOT_CARDINALITY_THRESHOLD
)
print(f"One-hot encoded columns ({len(one_hot_encoded_columns)}): {one_hot_encoded_columns}")
print(f"Label/ordinal encoded columns ({len(label_encoded_columns)}): {label_encoded_columns}")
print(f"\nFeature count after encoding: {X.shape[1]}")


One-hot encoded columns (25): ['1st_Road_Class', 'Day_of_Week', 'Junction_Control', 'Road_Surface_Conditions', 'InScotland', 'Age_Band_of_Driver', 'Driver_Home_Area_Type', 'Journey_Purpose_of_Driver', 'Junction_Location', 'Propulsion_Code', 'Sex_of_Driver', 'Towing_and_Articulation', 'Vehicle_Leaving_Carriageway', 'Was_Vehicle_Left_Hand_Drive', 'X1st_Point_of_Impact', 'Month', 'Time_of_Day', 'Speed_Category', 'Road_Category', 'Junction_Group', 'Weather_Group', 'Light_Group', 'Urban_Rural_Group', 'Vehicle_Type_Group', 'Vehicle_Class']
Label/ordinal encoded columns (1): ['Vehicle_Manoeuvre']

Feature count after encoding: 154


In [13]:
# Track which of the FINAL columns in X are categorical in nature (one-hot
# dummies or label-encoded codes) versus genuinely continuous/numeric, for use
# in Section 7's SMOTENC step, which requires this distinction explicitly.
one_hot_expanded_columns = [
    col for col in X.columns
    if any(col.startswith(f"{base}_") for base in one_hot_encoded_columns)
]

BINARY_INDICATOR_COLUMNS = [
    col for col in ["Is_Weekend", "Is_Night", "High_Speed_Road", "Urban_Area"] if col in X.columns
]

categorical_like_columns = list(
    dict.fromkeys(one_hot_expanded_columns + label_encoded_columns + BINARY_INDICATOR_COLUMNS)
)

print(f"Total categorical-like columns (for SMOTENC): {len(categorical_like_columns)}")


Total categorical-like columns (for SMOTENC): 139


In [14]:
X.head()

,Did_Police_Officer_Attend_Scene_of_Accident,Number_of_Vehicles,Pedestrian_Crossing-Human_Control,Pedestrian_Crossing-Physical_Facilities,Speed_limit,Year_accident,Age_of_Vehicle,Engine_Capacity_.CC.,Vehicle_Location.Restricted_Lane,Vehicle_Manoeuvre,Year_vehicle,Year,Day,Quarter,Is_Weekend,Hour,Minute,Is_Night,High_Speed_Road,Urban_Area,1st_Road_Class_A(M),1st_Road_Class_B,1st_Road_Class_C,1st_Road_Class_Motorway,1st_Road_Class_Unclassified,Day_of_Week_Monday,Day_of_Week_Saturday,Day_of_Week_Sunday,Day_of_Week_Thursday,Day_of_Week_Tuesday,Day_of_Week_Wednesday,Junction_Control_Auto traffic signal,Junction_Control_Data missing or out of range,Junction_Control_Give way or uncontrolled,Junction_Control_Not at junction or within 20 metres,Junction_Control_Stop sign,Road_Surface_Conditions_Dry,Road_Surface_Conditions_Flood over 3cm. deep,Road_Surface_Conditions_Frost or ice,Road_Surface_Conditions_Snow,Road_Surface_Conditions_Wet or damp,InScotland_Yes,Age_Band_of_Driver_11 - 15,Age_Band_of_Driver_16 - 20,Age_Band_of_Driver_21 - 25,Age_Band_of_Driver_26 - 35,Age_Band_of_Driver_36 - 45,Age_Band_of_Driver_46 - 55,Age_Band_of_Driver_56 - 65,Age_Band_of_Driver_6 - 10,Age_Band_of_Driver_66 - 75,Age_Band_of_Driver_Data missing or out of range,Age_Band_of_Driver_Over 75,Age_Band_of_Driver_Unknown,Driver_Home_Area_Type_Rural,Driver_Home_Area_Type_Small town,Driver_Home_Area_Type_Unknown,Driver_Home_Area_Type_Urban area,Journey_Purpose_of_Driver_Data missing or out of range,Journey_Purpose_of_Driver_Journey as part of work,Journey_Purpose_of_Driver_Not known,Journey_Purpose_of_Driver_Other,Journey_Purpose_of_Driver_Other/Not known (2005-10),Journey_Purpose_of_Driver_Pupil riding to/from school,Journey_Purpose_of_Driver_Taking pupil to/from school,Journey_Purpose_of_Driver_Unknown,Junction_Location_Cleared junction or waiting/parked at junction exit,Junction_Location_Data missing or out of range,Junction_Location_Entering from slip road,Junction_Location_Entering main road,Junction_Location_Entering roundabout,Junction_Location_Leaving main road,Junction_Location_Leaving roundabout,Junction_Location_Mid Junction - on roundabout or on main road,Junction_Location_Not at or within 20 metres of junction,...,Propulsion_Code_Gas Diesel,Propulsion_Code_Gas/Bi-fuel,Propulsion_Code_Heavy oil,Propulsion_Code_Hybrid electric,Propulsion_Code_New fuel technology,Propulsion_Code_Petrol,Propulsion_Code_Petrol/Gas (LPG),Propulsion_Code_Steam,Propulsion_Code_Unknown,Sex_of_Driver_Female,Sex_of_Driver_Male,Sex_of_Driver_Not known,Sex_of_Driver_Unknown,Towing_and_Articulation_Caravan,Towing_and_Articulation_Data missing or out of range,Towing_and_Articulation_Double or multiple trailer,Towing_and_Articulation_No tow/articulation,Towing_and_Articulation_Other tow,Towing_and_Articulation_Single trailer,Towing_and_Articulation_Unknown,Vehicle_Leaving_Carriageway_Did not leave carriageway,Vehicle_Leaving_Carriageway_Nearside,Vehicle_Leaving_Carriageway_Nearside and rebounded,Vehicle_Leaving_Carriageway_Offside,Vehicle_Leaving_Carriageway_Offside - crossed central reservation,Vehicle_Leaving_Carriageway_Offside and rebounded,Vehicle_Leaving_Carriageway_Offside on to central reservation,Vehicle_Leaving_Carriageway_Offside on to centrl res + rebounded,Vehicle_Leaving_Carriageway_Straight ahead at junction,Vehicle_Leaving_Carriageway_Unknown,Was_Vehicle_Left_Hand_Drive_No,Was_Vehicle_Left_Hand_Drive_Unknown,Was_Vehicle_Left_Hand_Drive_Yes,X1st_Point_of_Impact_Data missing or out of range,X1st_Point_of_Impact_Did not impact,X1st_Point_of_Impact_Front,X1st_Point_of_Impact_Nearside,X1st_Point_of_Impact_Offside,X1st_Point_of_Impact_Unknown,Month_August,Month_December,Month_February,Month_January,Month_July,Month_June,Month_March,Month_May,Month_November,Month_October,Month_September,Time_of_Day_Evening,Time_of_Day_Morning,Time_of_Day_Night,Speed_Category_40-50,Speed_Category_60+,Road_Category_Minor Road,Road_Category_Other,Road_Category_Roundabout,Junction_Group_Not at J

**Interpretation**

- **What was done:** Every categorical column was automatically detected and encoded
  based on its cardinality — low-cardinality columns via one-hot encoding, high-
  cardinality columns via ordinal/label encoding — with no manual, per-column
  hardcoding of the encoding choice.
- **Why it matters:** This dual strategy keeps the feature set usable by both linear
  models (which benefit from one-hot's lack of artificial ordering on nominal
  categories) and tree-based models (which handle label-encoded high-cardinality
  columns efficiently without an explosion of sparse one-hot columns).
- **Why not target encoding:** Target-based encodings (e.g., mean target encoding)
  would need to be fit only on the training partition to avoid leakage — since this
  notebook encodes before the train-test split (per the task's specified order), only
  target-independent encodings (one-hot, ordinal) are used here, which are safe to
  apply to the full dataset.


---
## 6. Handle Class Imbalance

**Purpose:** Analyze the target distribution's imbalance in detail and determine the
most appropriate handling strategy — **without** applying any resampling yet.

> **Important methodological note:** Although this task list numbers class-imbalance
> handling (Section 6) before the train-test split (Section 7), any resampling
> technique (such as SMOTE/SMOTENC) must only ever be **fit and applied to the training
> set, after splitting** — never to the full dataset beforehand. Resampling before
> splitting would let synthetic samples derived from what becomes test data (or their
> near-duplicate neighbors) leak into training, and would let information about the
> eventual test set's class balance influence how synthetic training samples are
> generated. **This notebook therefore analyzes and discusses imbalance here in Section
> 6, and performs the actual resampling in Section 7, immediately after the split and
> strictly on the training partition only** — preserving both the intent of the task
> list and correct ML methodology.


In [15]:
imbalance_ratio = target_counts.max() / target_counts.min()

print("Target Class Distribution:")
print(target_summary)
print()
print(f"Imbalance Ratio (majority : minority) = {imbalance_ratio:.1f} : 1")


Target Class Distribution:
  Severity    Count  Percentage (%)
0   Slight  2315841           85.27
1  Serious   364619           13.43
2    Fatal    35480            1.31

Imbalance Ratio (majority : minority) = 65.3 : 1


**Discussion — Suitable Approaches for This Imbalance**

Given the class distribution above (typically `Slight` as the large majority, with
`Serious` and especially `Fatal` as small minority classes), several complementary
strategies are appropriate:

1. **Class-weighted loss functions** (e.g., `class_weight="balanced"` in scikit-learn
   estimators) — a simple, low-risk first approach that requires no data duplication or
   synthesis, applied at model-training time (Phase 7, not here).
2. **Resampling** — oversampling the minority classes (e.g., SMOTE-family methods) or
   undersampling the majority class, applied **only to the training set**, to give the
   model more balanced exposure to minority-class patterns during learning.
3. **Threshold / decision-boundary tuning** and **metric selection** — using macro
   F1-score, recall on minority classes, or a confusion matrix rather than raw accuracy
   to evaluate models, since accuracy alone is misleading under strong imbalance
   (deferred to the model evaluation phase).

**Why SMOTENC (not plain SMOTE) is used here:** Since Section 5 encoded categorical
columns into one-hot binary and label-encoded integer columns, the feature space is now
predominantly categorical-in-nature rather than purely continuous. Standard SMOTE
linearly interpolates between neighboring samples in continuous space, which would
generate **nonsensical fractional values** for one-hot/label-encoded categorical
columns (e.g., a one-hot flag of `0.37`). **SMOTENC** (SMOTE for Nominal and
Continuous features, from `imbalanced-learn`) is designed specifically for this mixed
setting: it correctly treats the categorical-like columns identified in Section 5 as
discrete during interpolation, while still smoothly interpolating genuinely continuous
columns. Given the meaningful imbalance ratio observed above, resampling **is**
appropriate here and is implemented with SMOTENC in Section 7, applied strictly to the
training set only.


---
## 7. Train-Test Split

**Purpose:** Create an 80/20 stratified train-test split (`random_state=42`), then
apply SMOTENC resampling to the **training set only**, immediately after splitting —
never to the test set, which must remain a faithful, untouched sample of the true
population for honest evaluation in later phases.


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape : {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape : {y_test.shape}")


X_train shape: (2172752, 154)
X_test shape : (543188, 154)
y_train shape: (2172752,)
y_test shape : (543188,)


In [17]:
print("Training set class distribution (before resampling):")
print(y_train.value_counts(normalize=True).mul(100).round(2))
print()
print("Test set class distribution:")
print(y_test.value_counts(normalize=True).mul(100).round(2))


Training set class distribution (before resampling):
Accident_Severity
Slight    85.27
Serious   13.43
Fatal      1.31
Name: proportion, dtype: float64

Test set class distribution:
Accident_Severity
Slight    85.27
Serious   13.43
Fatal      1.31
Name: proportion, dtype: float64


**Interpretation**

- The `stratify=y` parameter ensures both the training and test sets preserve the same
  class proportions as the full dataset — critical under strong imbalance, where a
  non-stratified split could leave very few (or zero) `Fatal` examples in the test set
  purely by chance.
- `random_state=42` guarantees this split is exactly reproducible across notebook runs.


In [18]:
if IMBLEARN_AVAILABLE:
    categorical_feature_indices = [
        X_train.columns.get_loc(col) for col in categorical_like_columns if col in X_train.columns
    ]

    smote_nc = SMOTENC(
        categorical_features=categorical_feature_indices,
        random_state=RANDOM_STATE,
    )

    X_train_resampled, y_train_resampled = smote_nc.fit_resample(X_train, y_train)

    print("SMOTENC resampling applied to the training set only.")
    print(f"Training shape before resampling: {X_train.shape}")
    print(f"Training shape after resampling : {X_train_resampled.shape}")
    print()
    print("Training set class distribution (after SMOTENC resampling):")
    print(y_train_resampled.value_counts(normalize=True).mul(100).round(2))

    X_train, y_train = X_train_resampled, y_train_resampled
else:
    print(
        "[SKIPPED] SMOTENC resampling was not applied because 'imbalanced-learn' is "
        "not installed in this environment. The training set below retains its "
        "original class imbalance; install 'imbalanced-learn' and re-run this cell "
        "to apply resampling."
    )


[SKIPPED] SMOTENC resampling was not applied because 'imbalanced-learn' is not installed in this environment. The training set below retains its original class imbalance; install 'imbalanced-learn' and re-run this cell to apply resampling.


**Interpretation**

- **What was done:** SMOTENC was fit and applied **only** on `X_train`/`y_train`,
  generating synthetic minority-class samples that respect the categorical nature of
  the encoded columns identified in Section 5, bringing all classes closer to balance
  in the training set.
- **Why the test set is untouched:** `X_test`/`y_test` remain exactly as split above,
  preserving the real-world class distribution so that any future model evaluation
  reflects genuine, unbiased performance on realistic data — synthetic samples must
  never appear in an evaluation set.
- **Impact on prediction:** A more balanced training set gives future models
  meaningfully more exposure to minority-class (`Serious`/`Fatal`) patterns during
  learning, which is expected to improve recall on those classes relative to training
  on the original imbalanced data — to be verified empirically once modeling begins.


---
## 8. Save Prepared Data

**Purpose:** Persist the final, model-ready `X_train`, `X_test`, `y_train`, and
`y_test` to `Dataset/processed/ml_ready/`, creating the destination folder
automatically if it does not already exist.


In [19]:
ML_READY_DIR = Path("..") / "Dataset" / "processed" / "ml_ready"
ML_READY_DIR.mkdir(parents=True, exist_ok=True)

try:
    X_train.to_csv(ML_READY_DIR / "X_train.csv", index=False)
    X_test.to_csv(ML_READY_DIR / "X_test.csv", index=False)
    y_train.to_csv(ML_READY_DIR / "y_train.csv", index=False, header=[TARGET_COLUMN])
    y_test.to_csv(ML_READY_DIR / "y_test.csv", index=False, header=[TARGET_COLUMN])

    print(f"Prepared data saved successfully to: {ML_READY_DIR.resolve()}")
    print("  - X_train.csv")
    print("  - X_test.csv")
    print("  - y_train.csv")
    print("  - y_test.csv")
except OSError as save_error:
    print(f"[ERROR] Failed to save prepared data: {save_error}")
    raise


Prepared data saved successfully to: C:\Users\Stalin\OneDrive\Desktop\draftrajproject\road-accident-severity-prediction\road-accident-severity-prediction\Dataset\processed\ml_ready
  - X_train.csv
  - X_test.csv
  - y_train.csv
  - y_test.csv


**Interpretation**

- `X_train`/`y_train` reflect the **SMOTENC-resampled, class-balanced** training data
  (when `imbalanced-learn` is available), while `X_test`/`y_test` reflect the
  **original, untouched, realistically imbalanced** test data — this distinction is
  intentional and important for the next phase to use correctly.
- The output directory is created programmatically with `Path.mkdir(parents=True,
  exist_ok=True)`, so this cell runs successfully whether or not
  `Dataset/processed/ml_ready/` already exists.


---
## 9. Feature Summary

**Purpose:** Summarize every retained conceptual feature — its type, the encoding
applied, and the reason for its inclusion — as a single reference table for the
modeling phase and the final research report.


In [20]:
FEATURE_TYPE_DESCRIPTIONS = {
    "Number_of_Vehicles": ("Numerical", "None", "Count of vehicles involved; directly relevant to collision complexity."),
    "Speed_limit": ("Numerical", "None", "Posted speed limit; strong physical link to collision energy and severity."),
    "Year": ("Numerical", "None", "Calendar year; captures long-term trend effects."),
    "Month": ("Categorical (low cardinality)", "One-Hot", "Captures seasonal variation identified in EDA."),
    "Day": ("Numerical", "None", "Day-of-month component of the accident date."),
    "Quarter": ("Numerical", "None", "Coarser seasonal grouping than month."),
    "Day_of_Week": ("Categorical (low cardinality)", "One-Hot", "Captures weekday-specific traffic/risk patterns."),
    "Is_Weekend": ("Binary Indicator", "None (already 0/1)", "Flags weekend timing, linked to leisure-travel risk patterns."),
    "Hour": ("Numerical", "None", "Clock hour; supports fine-grained time-of-day patterns."),
    "Minute": ("Numerical", "None", "Clock minute; minor granularity supporting Hour/Time_of_Day."),
    "Time_of_Day": ("Categorical (low cardinality)", "One-Hot", "Readable time-of-day bucket linked to visibility/fatigue risk."),
    "Speed_Category": ("Categorical (low cardinality)", "One-Hot", "Readable speed band, reflecting the speed-severity link from EDA."),
    "Road_Category": ("Categorical (low cardinality)", "One-Hot", "Groups road type into Major/Minor/Roundabout for traffic-speed context."),
    "Junction_Group": ("Categorical (low cardinality)", "One-Hot", "Flags junction presence, a known collision-type driver."),
    "Junction_Control": ("Categorical (low cardinality)", "One-Hot", "Type of control at a junction; affects right-of-way risk."),
    "Weather_Group": ("Categorical (low cardinality)", "One-Hot", "Cleaned weather category, reducing sparsity in rare adverse conditions."),
    "Light_Group": ("Categorical (low cardinality)", "One-Hot", "Distinguishes daylight, lit darkness, and unlit darkness."),
    "Urban_Rural_Group": ("Categorical (low cardinality)", "One-Hot", "Standardized urban/rural label; one of the strongest known severity predictors."),
    "Vehicle_Age": ("Numerical", "None", "Age of the vehicle; older vehicles may lack modern safety features."),
    "Vehicle_Type_Group": ("Categorical (low cardinality)", "One-Hot", "Groups vehicle type by occupant-vulnerability profile."),
    "Vehicle_Class": ("Categorical (low cardinality)", "One-Hot", "Engine-size band, a proxy for vehicle size/mass and collision energy."),
    "Is_Night": ("Binary Indicator", "None (already 0/1)", "Flags unlit night-time driving, a known visibility-related risk window."),
    "High_Speed_Road": ("Binary Indicator", "None (already 0/1)", "Flags high-speed roads (>=60 mph), linked to collision-energy risk."),
    "Urban_Area": ("Binary Indicator", "None (already 0/1)", "Simple complement to the rural-severity relationship."),
    "Driver_IMD_Decile": ("Numerical", "None", "Socioeconomic deprivation decile of the driver's home area."),
    "1st_Road_Class": ("Categorical (low/high cardinality)", "One-Hot or Label (cardinality-dependent)", "Road classification (A/B/C/Unclassified), related to Road_Category."),
    "Propulsion_Code": ("Categorical (low cardinality)", "One-Hot", "Vehicle fuel/propulsion type."),
    "Sex_of_Driver": ("Categorical (low cardinality)", "One-Hot", "Recorded sex of the driver."),
    "Age_Band_of_Driver": ("Categorical (low cardinality)", "One-Hot", "Banded driver age group."),
}


def build_feature_summary(
    df: pd.DataFrame,
    one_hot_columns: list,
    label_columns: list,
) -> pd.DataFrame:
    '''
    Build a feature summary table for every conceptual column retained
    after Section 4's removals, describing its type, encoding, and
    rationale for inclusion.

    Args:
        df (pd.DataFrame): The dataframe of conceptual (pre-expansion) feature names
                            to summarize (evaluated just before one-hot expansion).
        one_hot_columns (list): Names of columns that were one-hot encoded.
        label_columns (list): Names of columns that were label/ordinal encoded.

    Returns:
        pd.DataFrame: The feature summary table.
    '''
    rows = []
    for col in df.columns:
        if col in one_hot_columns:
            encoding = "One-Hot Encoding"
        elif col in label_columns:
            encoding = "Ordinal / Label Encoding"
        else:
            encoding = "None (already numerical)"

        default_type, default_encoding, default_reason = FEATURE_TYPE_DESCRIPTIONS.get(
            col, ("Numerical" if pd.api.types.is_numeric_dtype(df[col]) else "Categorical", encoding, "Retained as a relevant predictor identified during feature engineering / EDA.")
        )

        rows.append(
            {
                "Feature Name": col,
                "Feature Type": default_type,
                "Encoding Used": encoding if col in one_hot_columns or col in label_columns else default_encoding,
                "Reason for Inclusion": default_reason,
            }
        )

    return pd.DataFrame(rows)


feature_summary_df = build_feature_summary(
    pre_encoding_columns_snapshot, one_hot_encoded_columns, label_encoded_columns
)
feature_summary_df


,Feature Name,Feature Type,Encoding Used,Reason for Inclusion
0,1st_Road_Class,Categorical (low/high cardinality),One-Hot Encoding,"Road classification (A/B/C/Unclassified), rela..."
1,Day_of_Week,Categorical (low cardinality),One-Hot Encoding,Captures weekday-specific traffic/risk patterns.
2,Did_Police_Officer_Attend_Scene_of_Accident,Numerical,None (already numerical),Retained as a relevant predictor identified du...
3,Junction_Control,Categorical (low cardinality),One-Hot Encoding,Type of control at a junction; affects right-o...
4,Number_of_Vehicles,Numerical,None,Count of vehicles involved; directly relevant ...
5,Pedestrian_Crossing-Human_Control,Numerical,None (already numerical),Retained as a relevant predictor identified du...
6,Pedestrian_Crossing-Physical_Facilities,Numerical,None (already numerical),Retained as a relevant predictor identified du...
7,Road_Surface_Conditions,Categorical,One-Hot Encoding,Retained as a relevant predictor identified du...
8,Speed_limit,Numerical,None,Posted speed limit; strong physical link to co...
9,Year_accident,Numerical,None (already numerical),Retained as a relevant predictor identified du...


**Interpretation**

- This table documents every conceptual feature retained after Section 4's removals,
  at the pre-encoding level (i.e., before one-hot expansion into multiple dummy
  columns), since that is the most readable and useful granularity for a research
  report or GitHub audience.
- Encoding is looked up programmatically from the actual lists produced in Section 5
  (`one_hot_encoded_columns`, `label_encoded_columns`), so this table always reflects
  what was truly done in this run rather than a hardcoded assumption.


---
## 10. Final Validation

**Purpose:** Print a final, consolidated validation summary of the prepared dataset —
training/testing shapes, target distributions, and feature count — confirming
readiness for the next phase.


In [21]:
print("=" * 60)
print("FINAL VALIDATION SUMMARY")
print("=" * 60)

print(f"\nTraining Shape (X_train) : {X_train.shape}")
print(f"Testing Shape (X_test)   : {X_test.shape}")
print(f"Feature Count            : {X_train.shape[1]}")

print("\nTraining Target Distribution:")
print(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nTesting Target Distribution:")
print(y_test.value_counts(normalize=True).mul(100).round(2))

print("\n" + "=" * 60)
print(f"Prepared data saved to: {ML_READY_DIR.resolve()}")
print("=" * 60)


FINAL VALIDATION SUMMARY

Training Shape (X_train) : (2172752, 154)
Testing Shape (X_test)   : (543188, 154)
Feature Count            : 154

Training Target Distribution:
Accident_Severity
Slight    85.27
Serious   13.43
Fatal      1.31
Name: proportion, dtype: float64

Testing Target Distribution:
Accident_Severity
Slight    85.27
Serious   13.43
Fatal      1.31
Name: proportion, dtype: float64

Prepared data saved to: C:\Users\Stalin\OneDrive\Desktop\draftrajproject\road-accident-severity-prediction\road-accident-severity-prediction\Dataset\processed\ml_ready


**Interpretation**

- This final summary confirms the dataset is fully prepared for machine learning: a
  balanced (or original, if `imbalanced-learn` was unavailable) training set, an
  untouched, realistically distributed test set, and a fully numeric, encoded feature
  matrix with no remaining identifier, leakage, or redundant columns.

### Next Steps

The next notebook (`06_Model_Development.ipynb`, Phase 7 of the project roadmap) will
load the arrays saved in `Dataset/processed/ml_ready/` to train, tune, and evaluate
machine learning models — none of which is performed here, consistent with this
notebook's feature-selection-and-preparation-only scope.
